# 1. Pré-processamento de dados

De sequência de DNA, que é texto, para tensor, que é número.

Ao final desta seção teremos um array de forma `(55001, 201, 4)`, com cada
número carregando um significado biológico preciso.

## 1.1. Dados do genoma do cupuaçu

O cupuaçu (*Theobroma grandiflorum*) é uma árvore frutífera amazônica, parente
próxima do cacau.

<img src="images/cupuacu.jpg" width="620">

Genoma montado telômero-a-telômero, publicado em
[Alves et al., *GigaScience* 13:giae027, 2024](https://pubmed.ncbi.nlm.nih.gov/38837946/). Os dados ficam no
GigaDB, no [conjunto 102523](https://gigadb.org/dataset/102523), distribuídos em dois arquivos:

- um **FASTA** com a sequência dos 10 cromossomos, 424 Mb no total;
- um **GFF3** com a anotação, que diz onde estão os genes, os éxons e os
  íntrons.

## 1.2. Extração das janelas

Esses dois arquivos não servem direto para treinar. Antes, eles passam por um processo que recorta pedaços de sequência e decide quais são sítio doador.

Esse processo rodou uma vez, fora da aula, e o resultado é o arquivo que a próxima subseção carrega. Vale entender como ele funciona, porque é dali que saem os rótulos: a afirmação de que uma janela é ou não um sítio real vem daqui, e não de outro lugar.

O **sítio doador** é a fronteira onde um íntron começa. A subseção 1.4 desenvolve a biologia; por ora basta saber que é uma posição específica do genoma, e que a anotação diz onde cada uma está.

<img src="images/extraction.png" width="1500">

O diagrama percorre as quatro etapas da esquerda para a direita.

Dois números não aparecem nele, e valem registro. Primeiro, a proporção de negativos por positivo é 10:1 no treino, mas sobe para a natural 50:1 em validação e teste, o que é assunto da seção 2. Segundo, o arquivo final guarda
as janelas como texto, e não como tensor: a conversão para one-hot é a subseção 1.6, e é conteúdo de aula.

O código que faz tudo isso está aqui:

[`scripts/extract.py`](https://github.com/hellsdeur/gendl/blob/main/scripts/extract.py)

São cerca de 130 linhas de processamento, mais as verificações. Ele não roda na aula porque precisa ler os 424 Mb do genoma, mas vale a leitura depois: a conversão de coordenada entre as duas fitas é o trecho com maior risco de erro silencioso de todo o projeto.

## 1.3. Carregamento dos dados

Com o processo entendido, o arquivo deixa de ser uma caixa-preta.

A célula abaixo baixa os 15 MB na primeira execução. Rodando local, ela só usa o arquivo que já está na pasta.

In [ ]:
import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

URL_DADOS = "https://github.com/hellsdeur/gendl/raw/refs/heads/main/data/splice_donor.npz"
CAMINHO_DADOS = Path("data/splice_donor.npz")

# Rodando local o arquivo já está aqui. No Colab, baixa na primeira execução.
if not CAMINHO_DADOS.exists():
    print("Baixando splice_donor.npz, 15 MB...")
    CAMINHO_DADOS.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(URL_DADOS, CAMINHO_DADOS)

# Um .npz é um zip, e todo zip começa com os bytes "PK". Se o download trouxe
# outra coisa, como uma página de erro, é melhor parar aqui com uma mensagem
# clara do que quebrar adiante com um erro do numpy.
if CAMINHO_DADOS.read_bytes()[:2] != b"PK":
    CAMINHO_DADOS.unlink()
    raise RuntimeError(f"O download não trouxe um .npz. Confira a URL:\n{URL_DADOS}")

dados = np.load(CAMINHO_DADOS)

# O .npz carrega junto um dicionário de metadados, gravado pelo script de
# extração. É de lá que vêm todos os números citados nesta seção: nenhum
# deles está digitado à mão no notebook.
meta = json.loads(str(dados["meta"]))
genoma = meta["genome_stats"]


def br(numero, casas=0):
    """Formata no padrão brasileiro: 1234567 -> 1.234.567 e 9.09 -> 9,09

    Milhar com ponto, decimal com vírgula. Sem isso a mesma saída usaria
    ponto para as duas coisas, e "55.001" ao lado de "9.09" fica ambíguo.
    """
    texto = f"{numero:,.{casas}f}"
    return texto.replace(",", "\x00").replace(".", ",").replace("\x00", ".")


print(f"Pares de base:      {br(genoma['genome_bp'])}")
print(f"Cromossomos:        {genoma['n_chromosomes']}")
print(f"Genes anotados:     {br(genoma['annotated_genes'])}")
print(f"Íntrons anotados:   {br(genoma['annotated_introns'])}")

## 1.4. Problema de encontrar sítios doadores em genomas

Um gene não vira proteína diretamente:

1. o gene é transcrito em RNA;
2. esse RNA tem **éxons**, que permanecem, e **íntrons**, que são removidos;
3. cortar os íntrons e emendar os éxons é o **splicing**.

<img src="images/genoma.png" width="900">

O **sítio doador** é a fronteira onde um íntron começa, e é ele que vamos prever.

Quase todo íntron começa com `GT`. O problema é que `GT` é um par de bases banal, e ele aparece em qualquer lugar do genoma.

<img src="images/genoma-2.png" width="960">

Seis `GT` em um trecho curto, e só um deles é sítio doador. Quantos existem no genoma inteiro?

In [ ]:
# Contagens sobre o genoma INTEIRO, calculadas pelo script de extração.
grupos = [
    ("Todos os GT\ndo genoma", genoma["gt_genome_wide"]),
    ("GT dentro\nde genes", genoma["gt_in_gene_bodies"]),
    ("Sítios doadores\nreais", genoma["donor_sites"]),
]
rotulos = [nome for nome, _ in grupos]
valores = [valor for _, valor in grupos]

# Rampa sequencial de um único tom: os três grupos são subconjuntos aninhados
# da mesma medida, do mais geral (claro) ao mais específico (escuro).
CORES = ["#86b6ef", "#2a78d6", "#104281"]
SUPERFICIE, TINTA, MUTED, GRADE = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"

figura, eixos = plt.subplots(1, 2, figsize=(12, 3.4), facecolor=SUPERFICIE)
posicao = range(len(grupos))

for indice, eixo in enumerate(eixos):
    eixo.barh(posicao, valores, color=CORES, height=0.62)
    eixo.set_yticks(posicao)
    eixo.set_yticklabels(rotulos, fontsize=9, color=TINTA)
    eixo.invert_yaxis()
    eixo.set_facecolor(SUPERFICIE)
    eixo.xaxis.grid(True, color=GRADE, linewidth=0.8)
    eixo.set_axisbelow(True)
    for lado in ("top", "right", "left"):
        eixo.spines[lado].set_visible(False)
    eixo.spines["bottom"].set_color(GRADE)
    eixo.tick_params(colors=MUTED, labelsize=8)

eixos[0].set_title("Escala linear", fontsize=10, color=TINTA, loc="left")
eixos[0].set_xlabel("ocorrências", fontsize=8, color=MUTED)
eixos[0].set_xlim(0, max(valores) * 1.28)

eixos[1].set_xscale("log")
eixos[1].set_xlim(1e4, 1e8)
eixos[1].set_title("Escala logarítmica", fontsize=10, color=TINTA, loc="left")
eixos[1].set_xlabel("ocorrências (log)", fontsize=8, color=MUTED)

# Os dois painéis rotulam coisas diferentes, para não repetir informação:
# à esquerda a proporção, à direita a contagem absoluta.
total = max(valores)
for indice, valor in enumerate(valores):
    eixos[0].text(valor + total * 0.02, indice,
                  f"{valor / total * 100:.2f}%".replace(".", ","),
                  va="center", fontsize=9, color=TINTA)
    eixos[1].text(valor * 1.25, indice, br(valor), va="center",
                  fontsize=9, color=TINTA)

figura.suptitle("Quantos GT existem, e quantos são sítio de splice",
                fontsize=12, color=TINTA, x=0.01, ha="left")
figura.tight_layout()
plt.show()

cobertura = genoma["donor_sites"] / genoma["annotated_introns"] * 100
print(f"Íntrons que começam com GT: {br(genoma['donor_sites'])} de "
      f"{br(genoma['annotated_introns'])}, ou {br(cobertura, 2)}%")
print()

razao_genoma = genoma["gt_genome_wide"] / genoma["donor_sites"]
razao_genes = genoma["gt_in_gene_bodies"] / genoma["donor_sites"]
print(f"No genoma inteiro:  1 sítio real a cada {razao_genoma:.0f} GT")
print(f"Dentro de genes:    1 sítio real a cada {razao_genes:.0f} GT")

Os mesmos três números, em duas escalas:

- linear: os sítios reais somem, porque são **0,29%** de todos os `GT`;
- logarítmica: ficam legíveis, mas o abismo entre as barras desaparece.

Guarde esse par de painéis. Na seção 2 o mesmo fenômeno reaparece, com duas métricas no lugar de duas escalas.

> Achar `GT` é trivial. Decidir qual `GT` é sítio doador é o problema, e a única informação disponível para decidir é a sequência ao redor.

## 1.5. Janelas de sequências genômicas

Um modelo não recebe o genoma inteiro, e sim **janelas** de tamanho fixo. As deste dataset têm 201 bases, ancoradas em um `GT`:

<img src="images/window.png" width="900">

O `GT` fica sempre na posição 100, tanto nos positivos quanto nos negativos. Se os negativos fossem trechos aleatórios do genoma, bastaria ao modelo procurar `GT`. Ancorados todos no mesmo lugar, ele precisa decidir pelo contexto.

In [ ]:
X_treino = dados["X_train"]   # sequências, ainda como texto
y_treino = dados["y_train"]   # 1 = sítio doador real, 0 = não é

# As sequências estão guardadas como bytes; .decode() devolve texto.
primeira = X_treino[0].decode()

print("Uma janela crua:")
print(primeira)
print()
print(f"Comprimento:        {len(primeira)} bases")
print(f"Posição 100 e 101:  {primeira[100:102]}")
print(f"Rótulo:             {y_treino[0]}")

Duas janelas do conjunto de treino, na região central. Uma é um sítio doador real, a outra não.

<img src="images/window-2.png" width="960">

Qual das duas?

Seis janelas que são sítio doador e seis que não são, mostradas na região central.

O que o primeiro grupo tem que o segundo não tem?

In [ ]:
POSICAO_DOADOR = meta["donor_offset"]   # 100


def mostrar_centro(sequencia, margem=10):
    """Recorta a região central da janela e separa o GT com colchetes."""
    esquerda = sequencia[POSICAO_DOADOR - margem:POSICAO_DOADOR]
    direita = sequencia[POSICAO_DOADOR + 2:POSICAO_DOADOR + 2 + margem]
    meio = sequencia[POSICAO_DOADOR:POSICAO_DOADOR + 2]
    return f"{esquerda}[{meio}]{direita}"


print("SÍTIOS DOADORES REAIS")
for janela in X_treino[y_treino == 1][:6]:
    print("   ", mostrar_centro(janela.decode()))

print("\nNÃO SÃO SÍTIO")
for janela in X_treino[y_treino == 0][:6]:
    print("   ", mostrar_centro(janela.decode()))

Você provavelmente viu `AG` logo antes do `GT` em várias do primeiro grupo, e `AAG` logo depois em algumas. É o consenso do sítio doador, escrito na literatura como `MAG|GTAAGT`.

Isso também responde à pergunta das duas janelas: a **A** é o sítio real, com `AG` antes e `GTAAGT` depois. A **B** não tem nenhum dos dois.

Mas o padrão não está em todas. Melhor medir do que confiar na impressão.

In [ ]:
def frequencia(janelas, trecho, inicio):
    """Fração das janelas que contêm `trecho` a partir da posição `inicio`."""
    fim = inicio + len(trecho)
    return np.mean([j.decode()[inicio:fim] == trecho for j in janelas]) * 100


positivas = X_treino[y_treino == 1]
negativas = X_treino[y_treino == 0]

print(f"{'padrão':<22} {'nos sítios':>11} {'nos não-sítios':>15}")
print("-" * 50)
for descricao, trecho, inicio in [
    ("AG logo antes do GT", "AG", POSICAO_DOADOR - 2),
    ("GTA", "GTA", POSICAO_DOADOR),
    ("GTAAG", "GTAAG", POSICAO_DOADOR),
    ("GTAAGT (consenso)", "GTAAGT", POSICAO_DOADOR),
]:
    print(f"{descricao:<22} {frequencia(positivas, trecho, inicio):9.1f}% "
          f"{frequencia(negativas, trecho, inicio):14.1f}%")

O problema da aula, agora em números:

- exigir o consenso completo `GTAAGT` perderia 91 de cada 100 sítios reais;
- `AG` antes do `GT` acha metade dos sítios, mas também aparece em 5,9% dos não-sítios, que são 50 vezes mais numerosos.

Nenhuma regra fixa resolve. Todo padrão visível é uma tendência, não um critério.

É por isso que, daqui em diante, o problema deixa de ser atacado por regra e passa a ser atacado por modelos que aprendem pesos a partir dos exemplos.

## 1.6. One-hot encoding

Redes neurais operam sobre números, não sobre letras.

A tradução ingênua `A=0, C=1, G=2, T=3` está errada: ela inventa uma ordem e uma distância que não existem na biologia. Afirma que `T` é três vezes `C`, e que `A` está mais longe de `T` do que de `C`. O modelo acreditaria.

No **one-hot** cada base vira um vetor com um único 1, o que deixa as quatro equidistantes entre si:

| base | vetor |
|---|---|
| A | `[1, 0, 0, 0]` |
| C | `[0, 1, 0, 0]` |
| G | `[0, 0, 1, 0]` |
| T | `[0, 0, 0, 1]` |

Uma janela de 201 bases vira, então, uma matriz `(201, 4)`.

### 1.6.1. A função escrita à mão

A função abaixo recebe uma sequência de DNA como texto e devolve a matriz
`(comprimento, 4)`.

Ela percorre a sequência base por base e marca com `1.0` a coluna
correspondente. `BASES.index("C")` devolve `1`, que é a coluna do `C`.

In [ ]:
BASES = "ACGT"


def one_hot(sequencia):
    matriz = np.zeros((len(sequencia), 4), dtype=np.float32)
    for posicao, base in enumerate(sequencia):
        matriz[posicao, BASES.index(base)] = 1.0
    return matriz


# Com as quatro bases em ordem, o resultado é a matriz identidade.
print(one_hot("ACGT"))

O laço acima serve para entender. Para converter os 55 mil exemplos de uma vez ele seria lento demais.

A versão vetorizada faz a mesma operação de uma vez só, com numpy.

In [ ]:
def one_hot_lote(sequencias):
    """Converte um array inteiro de sequências para one-hot, de uma vez.

    Devolve um array de forma (n_exemplos, comprimento, 4).
    """
    # Cada caractere vira seu código ASCII: 'A' -> 65, 'C' -> 67, ...
    codigos = sequencias.view(np.uint8).reshape(len(sequencias), -1)

    # Tabela que traduz código ASCII -> coluna do one-hot.
    tabela = np.zeros(256, dtype=np.uint8)
    for coluna, base in enumerate(BASES):
        tabela[ord(base)] = coluna

    return np.eye(4, dtype=np.float32)[tabela[codigos]]


X_treino_oh = one_hot_lote(X_treino)
print("Forma do conjunto de treino:", X_treino_oh.shape)

## 1.7. Formato dos dados

Os três eixos do array, que é a leitura que se repete no resto da aula:

In [ ]:
n_exemplos, comprimento, n_bases = X_treino_oh.shape

print(f"({n_exemplos}, {comprimento}, {n_bases})")
print()
print(f"  eixo 0 = {n_exemplos:>6} janelas do conjunto de treino")
print(f"  eixo 1 = {comprimento:>6} posições dentro de cada janela")
print(f"  eixo 2 = {n_bases:>6} bases possíveis (A, C, G, T)")

Checagem de sanidade: se cada posição tem exatamente uma base, somar ao longo do último eixo tem que dar 1 em toda posição.

In [ ]:
somas = X_treino_oh.sum(axis=2)
print("Toda posição soma exatamente 1:", bool(np.all(somas == 1.0)))

# E a âncora: todo exemplo tem G na posição 100 e T na 101.
coluna_G, coluna_T = BASES.index("G"), BASES.index("T")
tem_G = np.all(X_treino_oh[:, POSICAO_DOADOR, coluna_G] == 1.0)
tem_T = np.all(X_treino_oh[:, POSICAO_DOADOR + 1, coluna_T] == 1.0)
print("Todo exemplo tem GT na posição 100:", bool(tem_G and tem_T))

Os três conjuntos já vêm separados no arquivo. A divisão foi feita por cromossomo, e não por sorteio. O porquê é assunto da seção 2.

In [ ]:
for nome in ("train", "val", "test"):
    rotulos = dados[f"y_{nome}"]
    positivos = int(rotulos.sum())
    print(f"{nome:>6}: {br(len(rotulos)):>9} exemplos, "
          f"{br(positivos):>6} positivos "
          f"({br(positivos / len(rotulos) * 100, 2):>5}%)")

Duas leituras da última coluna:

- validação e teste têm 1,96% de positivos, que é a proporção real do genoma: cerca de 1 sítio a cada 51 `GT` dentro de genes;
- treino tem 9,09% porque ali os negativos foram subamostrados de propósito, para o treino ficar mais barato.

Dá para baratear o treino. Não dá para baratear a avaliação sem mentir para si mesmo.

Guarde essa diferença. A seção 2 olha de perto o que significa treinar e avaliar com classes tão desiguais, e como dividir os dados sem enganar a si mesmo.

# 2. Balanceamento de classes

Duas propriedades deste dataset restringem tudo o que vem depois:

1. as classes são muito desiguais;
2. a divisão entre treino e teste não pode ser sorteada.

Nenhuma das duas é escolha de quem modela. São fatos do dado, e é melhor conhecê-los antes de treinar qualquer coisa.

## 2.1. Contagem de classes

Quantos sítios reais existem no conjunto de teste, e quantos `GT` que não são sítio?

In [ ]:
y_teste = dados["y_test"]

n_positivos = int(y_teste.sum())
n_negativos = int((y_teste == 0).sum())
taxa_positivos = y_teste.mean()

print(f"Positivos (sítio real):  {br(n_positivos):>9}")
print(f"Negativos (não é sítio): {br(n_negativos):>9}")
print(f"Razão:                   {br(n_negativos / n_positivos):>9} para 1")
print(f"Taxa de positivos:       {br(taxa_positivos * 100, 2):>8}%")

Essa razão não foi escolhida, foi medida. É a proporção de `GT` dentro de corpos de gene que não são sítio real, calculada no genoma inteiro na seção 1.

O dataset usa esse pool, e não o genoma todo, porque ele é o cenário de uso real: varrer um gene procurando onde ele faz splice. `GT` em região intergênica seria fácil demais, e o modelo passaria a separar as classes por composição da região, sem aprender nada sobre splicing.

In [ ]:
razao_genes = genoma["gt_in_gene_bodies"] / genoma["donor_sites"]
razao_genoma = genoma["gt_genome_wide"] / genoma["donor_sites"]

print(f"Pool usado (dentro de genes): {br(razao_genes):>4} negativos por positivo")
print(f"Pool do genoma inteiro:       {br(razao_genoma):>4} negativos por positivo")

Vale ver essa proporção desenhada. Cada quadrado abaixo é um `GT` candidato que o modelo vai ter que julgar.

In [ ]:
# Um retrato de 51 candidatos: a proporção que o modelo enfrenta.
n_quadrados = 51
colunas = 17
AZUL, NEUTRO = "#2a78d6", "#e1e0d9"

figura, eixo = plt.subplots(figsize=(9, 2.1), facecolor=SUPERFICIE)

for indice in range(n_quadrados):
    linha, coluna = divmod(indice, colunas)
    e_sitio = indice == 0
    eixo.add_patch(plt.Rectangle(
        (coluna, -linha), 0.86, 0.86,
        facecolor=AZUL if e_sitio else NEUTRO))

eixo.set_xlim(-0.3, colunas + 3.2)
eixo.set_ylim(-3.0, 1.2)
eixo.set_aspect("equal")
eixo.axis("off")
eixo.set_facecolor(SUPERFICIE)

eixo.text(colunas + 0.5, 0.43, "1 sítio doador real", va="center",
          fontsize=10, color=AZUL)
eixo.text(colunas + 0.5, -0.9, "50 candidatos que\nnão são sítio", va="center",
          fontsize=10, color=MUTED)

figura.suptitle("O que o modelo enfrenta a cada 51 candidatos",
                fontsize=12, color=TINTA, x=0.01, ha="left")
figura.tight_layout()
plt.show()

Repare que essa proporção vale para validação e teste, mas não para o treino.

O treino usa 10 negativos por positivo, e não 50. Subamostrar negativos deixa o treino mais barato sem mudar o que o modelo tem para aprender. Já a avaliação precisa da proporção real, senão o número que sai dela não descreve o uso.

In [ ]:
for conjunto in ("train", "val", "test"):
    rotulos = dados[f"y_{conjunto}"]
    positivos = int(rotulos.sum())
    negativos = len(rotulos) - positivos
    print(f"{conjunto:>6}: {br(negativos / positivos):>3} negativos por positivo "
          f"({br(positivos / len(rotulos) * 100, 2):>5}% de positivos)")

## 2.2. Split por cromossomo

Falta decidir quais janelas vão para treino e quais para teste. Sortear ao acaso seria o caminho natural, e seria errado aqui.

O genoma do cupuaçu passou por expansões de famílias gênicas, e o próprio paper enfatiza isso: existem genes duplicados, com sequência quase idêntica, em locais diferentes. Um sorteio aleatório coloca cópias do mesmo gene nos dois lados, e o modelo acerta no teste por ter decorado no treino.

A divisão usada separa cromossomos inteiros.

In [ ]:
for conjunto, cromossomos in meta["splits"].items():
    rotulos = dados[f"y_{conjunto}"]
    print(f"{conjunto:>6}: {', '.join(cromossomos):<40} "
          f"{br(len(rotulos)):>9} exemplos")

Separar por cromossomo reduz o vazamento, mas não elimina: genes duplicados também existem entre cromossomos diferentes.

In [ ]:
repetidas = meta["test_windows_seen_in_train"]
total_teste = len(dados["y_test"])

print(f"Janelas de teste idênticas a alguma do treino: "
      f"{repetidas} de {br(total_teste)} "
      f"({br(repetidas / total_teste * 100, 2)}%)")

Esse número é um limite inferior do vazamento real. Ele conta apenas janelas byte a byte idênticas, e parálogos costumam ser parecidos, não idênticos. Serve como ordem de grandeza, e como lembrete de que dividir por cromossomo é uma mitigação, não uma garantia.

Um último ponto sobre a origem dos rótulos, que vale para qualquer trabalho com dado genômico anotado.

Os sítios usados aqui vêm de anotação computacional apoiada em evidência transcriptômica real, e não de validação experimental sítio a sítio. Estamos treinando um modelo para reproduzir a saída de outro modelo. O teto de desempenho possível é o teto do anotador.

Com o dado entendido e dividido, a seção 3 constrói o primeiro modelo. É lá, com um score na mão, que o desbalanceamento desta seção vai cobrar seu preço: decidir se um modelo é bom exige escolher uma métrica, e essa escolha deixa de ser óbvia quando 98% do conjunto pertence a uma classe só.

# 3. Regressão logística

O primeiro modelo da aula, e a baseline contra a qual todo o resto vai ser comparado.

Ele também é o primeiro a produzir um score contínuo, e é por isso que a segunda metade desta seção monta o vocabulário de avaliação inteiro.

## 3.1. Achatamento do tensor

Um classificador linear não recebe matriz, recebe vetor. Então cada janela `(201, 4)` precisa virar uma lista de 804 números.

O que se perde nessa conta: cada par (posição, base) vira uma feature independente, e o modelo deixa de saber que a posição 40 é vizinha da 41.

Para um modelo linear isso não custa nada, porque ele já trataria as posições como independentes de qualquer forma. Guarde a observação: é exatamente esse buraco que a seção 4 vai preencher.

In [ ]:
X_treino_oh = one_hot_lote(dados["X_train"])
X_teste_oh = one_hot_lote(dados["X_test"])
y_treino = dados["y_train"]

# De (N, 201, 4) para (N, 804). O -1 deixa o numpy calcular a dimensão.
F_treino = X_treino_oh.reshape(len(X_treino_oh), -1)
F_teste = X_teste_oh.reshape(len(X_teste_oh), -1)

print("treino:", F_treino.shape)
print("teste: ", F_teste.shape)

## 3.2. Ajuste do modelo

A regressão logística aprende um peso para cada uma das 804 features, mais um intercepto. Ela soma tudo e passa o resultado por uma sigmoide, o que devolve um número entre 0 e 1.

In [ ]:
import time

from sklearn.linear_model import LogisticRegression

inicio = time.time()
modelo = LogisticRegression(max_iter=2000, n_jobs=-1)
modelo.fit(F_treino, y_treino)
segundos = time.time() - inicio

# Registro dos tempos de treino, para a comparação final da seção 5.
tempos = {"regressão logística": segundos}

print(f"Treinou em {br(segundos, 1)} s")
print(f"Parâmetros ajustados: {br(modelo.coef_.size + 1)}")

Treinou. Falta a pergunta difícil: treinou bem?

Responder isso exige escolher uma métrica, e é aí que o desbalanceamento da seção 2 cobra o preço.

## 3.3. Como mentir com métricas de avaliação

Antes de medir o modelo, vale medir dois classificadores que não aprenderam nada.

O primeiro responde "não é sítio" para tudo, sem olhar para o DNA.

In [ ]:
def acuracia(y_verdadeiro, y_predito):
    """Fração dos exemplos em que a predição bateu com o rótulo."""
    return (y_verdadeiro == y_predito).mean()


sempre_nao = np.zeros(len(y_teste), dtype=int)

print(f"Acurácia do 'sempre não': {br(acuracia(y_teste, sempre_nao) * 100, 2)}%")

Perto de 98% de acurácia, sem olhar para o DNA e sem aprender nada.

Agora o classificador oposto, que aceita todo `GT` como sítio.

In [ ]:
sempre_sim = np.ones(len(y_teste), dtype=int)

print(f"{'classificador':<18} {'acurácia':>9} {'sítios encontrados':>20}")
print("-" * 50)
for nome, predicao in [("sempre não", sempre_nao), ("todo GT é sítio", sempre_sim)]:
    encontrados = int(predicao[y_teste == 1].sum())
    print(f"{nome:<18} {br((predicao == y_teste).mean() * 100, 2):>8}% "
          f"{br(encontrados):>12} de {br(n_positivos)}")

Os dois são inúteis, e de formas opostas: um nunca encontra nada, o outro aponta tudo.

A acurácia premia o primeiro com 98% e pune o segundo com 2%. Na prática é o primeiro que é mais perigoso, porque um modelo que nunca encontra nada passa despercebido num relatório.

Acurácia não vai aparecer de novo neste notebook.

## 3.4. Precisão, revocação e PR-AUC

A acurácia mistura duas perguntas que precisam ficar separadas:

- precisão: dos `GT` que o modelo apontou como sítio, quantos eram mesmo?
- revocação: dos sítios reais que existem, quantos o modelo encontrou?

Nos dois classificadores triviais elas ficam em extremos opostos.

In [ ]:
from sklearn.metrics import precision_score, recall_score

print(f"{'classificador':<18} {'precisão':>9} {'revocação':>10}")
print("-" * 40)
for nome, predicao in [("sempre não", sempre_nao), ("todo GT é sítio", sempre_sim)]:
    p = precision_score(y_teste, predicao, zero_division=0)
    r = recall_score(y_teste, predicao, zero_division=0)
    print(f"{nome:<18} {br(p * 100, 2):>8}% {br(r * 100, 2):>9}%")

Um modelo útil precisa das duas ao mesmo tempo, e há uma troca entre elas: para encontrar mais sítios é preciso aceitar mais candidatos, o que derruba a precisão.

O score contínuo da regressão logística permite percorrer essa troca inteira. Cada limiar de corte dá um par (precisão, revocação), e o conjunto deles é a curva precisão-revocação. A área sob ela, PR-AUC, resume o modelo em um número.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

score_logistica = modelo.decision_function(F_teste)
pr_auc = average_precision_score(y_teste, score_logistica)

print(f"PR-AUC da regressão logística: {br(pr_auc, 4)}")
print(f"Piso trivial (taxa de positivos): {br(taxa_positivos, 4)}")

O piso merece atenção, porque é onde a intuição costuma falhar.

Em PR-AUC o piso trivial não é 0,5. É a taxa de positivos do conjunto, aqui 0,0196. Um modelo que sorteasse ao acaso ficaria em torno disso, e não na metade.

Quem chega acostumado com a AUC da curva ROC espera 0,5 e vai ler todos os números desta aula errado se isso não ficar claro agora.

## 3.5. ROC versus PR

A curva ROC é a métrica mais usada em classificação binária. Vale calcular as duas sobre exatamente as mesmas predições.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

roc_auc = roc_auc_score(y_teste, score_logistica)

print(f"ROC-AUC: {br(roc_auc, 3)}   (piso 0,5)")
print(f"PR-AUC:  {br(pr_auc, 3)}   (piso {br(taxa_positivos, 4)})")

Um número parece dizer "problema resolvido". O outro diz "dois terços do caminho". As predições são as mesmas.

In [ ]:
fpr, tpr, _ = roc_curve(y_teste, score_logistica)
precisao_curva, revocacao_curva, _ = precision_recall_curve(y_teste, score_logistica)

AZUL, CINZA = "#2a78d6", "#898781"

figura, eixos = plt.subplots(1, 2, figsize=(11, 4.2), facecolor=SUPERFICIE)

# Painel ROC: a diagonal é o acaso.
eixos[0].plot([0, 1], [0, 1], linestyle="--", color=CINZA, linewidth=1.2)
eixos[0].plot(fpr, tpr, color=AZUL, linewidth=2)
eixos[0].set_title(f"ROC, AUC = {br(roc_auc, 3)}",
                   fontsize=11, color=TINTA, loc="left")
eixos[0].set_xlabel("taxa de falso positivo", fontsize=9, color=MUTED)
eixos[0].set_ylabel("taxa de verdadeiro positivo", fontsize=9, color=MUTED)
eixos[0].text(0.55, 0.42, "acaso", fontsize=9, color=CINZA, rotation=32)

# Painel PR: o acaso é uma reta na altura da taxa de positivos.
eixos[1].axhline(taxa_positivos, linestyle="--", color=CINZA, linewidth=1.2)
eixos[1].plot(revocacao_curva, precisao_curva, color=AZUL, linewidth=2)
eixos[1].set_title(f"Precisão-revocação, AUC = {br(pr_auc, 3)}",
                   fontsize=11, color=TINTA, loc="left")
eixos[1].set_xlabel("revocação", fontsize=9, color=MUTED)
eixos[1].set_ylabel("precisão", fontsize=9, color=MUTED)
eixos[1].text(0.30, 0.10, f"linha tracejada = acaso ({br(taxa_positivos, 4)})",
              fontsize=9, color=CINZA)

for eixo in eixos:
    eixo.set_xlim(0, 1)
    eixo.set_ylim(0, 1)
    eixo.set_facecolor(SUPERFICIE)
    eixo.grid(True, color=GRADE, linewidth=0.8)
    eixo.set_axisbelow(True)
    for lado in ("top", "right"):
        eixo.spines[lado].set_visible(False)
    for lado in ("bottom", "left"):
        eixo.spines[lado].set_color(GRADE)
    eixo.tick_params(colors=MUTED, labelsize=8)

figura.suptitle("O mesmo modelo, as mesmas predições, duas métricas",
                fontsize=12, color=TINTA, x=0.01, ha="left")
figura.tight_layout()
plt.show()

A causa está no denominador de cada métrica.

A taxa de falso positivo, usada na curva ROC, divide os falsos positivos pelos 100.001 negativos do conjunto. Mil erros mal movem esse número. Já a precisão divide os mesmos mil erros pelo total de janelas apontadas, onde eles competem com apenas 2.000 positivos, e despenca.

Sempre que uma classe é rara, ROC-AUC é otimista. Daqui em diante, PR-AUC é a métrica principal deste notebook, e ROC-AUC aparece ao lado só para lembrar da diferença.

O contraste aqui ainda é brando, porque a logística é um modelo bom. Na seção 4 aparece um caso em que ROC-AUC diz 0,742 e PR-AUC diz 0,060.

## 3.6. Coeficientes e o que o modelo aprendeu

O modelo aprendeu 804 pesos. Remodelando de volta para `(201, 4)`, cada peso volta a ter endereço: uma posição da janela e uma base.

Peso positivo significa que aquela base, naquela posição, empurra a decisão para "é sítio". Peso negativo empurra para o contrário.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

pesos = modelo.coef_[0].reshape(201, 4)

# Recorte da região informativa, para o heatmap caber na tela.
INICIO, FIM = POSICAO_DOADOR - 10, POSICAO_DOADOR + 16
recorte = pesos[INICIO:FIM]
posicoes = np.arange(INICIO, FIM) - POSICAO_DOADOR

# Escala divergente: azul para peso negativo, vermelho para positivo, cinza no
# zero. Os limites são simétricos para que o cinza caia exatamente em zero.
DIVERGENTE = LinearSegmentedColormap.from_list(
    "peso", ["#2a78d6", "#f0efec", "#e34948"])
limite = np.abs(recorte).max()

figura, eixo = plt.subplots(figsize=(12, 2.4), facecolor=SUPERFICIE)
imagem = eixo.imshow(recorte.T, cmap=DIVERGENTE, vmin=-limite, vmax=limite,
                     aspect="auto")

eixo.set_yticks(range(4))
eixo.set_yticklabels(list(BASES), fontsize=10, color=TINTA)
eixo.set_xticks(range(len(posicoes)))
eixo.set_xticklabels([f"{p:+d}" for p in posicoes], fontsize=8, color=MUTED)
eixo.set_xlabel("posição relativa ao GT", fontsize=9, color=MUTED)
eixo.tick_params(length=0)

for lado in eixo.spines.values():
    lado.set_visible(False)

barra = figura.colorbar(imagem, ax=eixo, pad=0.01)
barra.set_label("peso", fontsize=9, color=MUTED)
barra.ax.tick_params(colors=MUTED, labelsize=8)
barra.outline.set_visible(False)

figura.suptitle("O que a regressão logística aprendeu, posição por posição",
                fontsize=12, color=TINTA, x=0.01, ha="left")
figura.tight_layout()
plt.show()

Duas leituras, e as duas valem.

A primeira: o consenso reapareceu. Peso alto para `G` em −1, para `A` em −2, `A` em +2, `G` em +4 e `T` em +5. É o `MAG|GTAAGT` que a seção 1 mediu contando à mão, agora reconstruído por um modelo que nunca recebeu uma linha de biologia. Ele só otimizou para separar as duas classes.

A segunda é mais sutil. Olhe as posições 0 e +1, que são o próprio `GT`.

In [ ]:
print("posição   peso de cada base")
for offset in (-1, 0, 1, 2):
    linha = pesos[POSICAO_DOADOR + offset]
    pesos_texto = "  ".join(f"{b}={br(w, 2):>5}" for b, w in zip(BASES, linha))
    print(f"  {offset:+3d}     {pesos_texto}")

Peso praticamente zero nas duas posições da âncora.

Faz sentido: toda janela do dataset tem `GT` ali, nos positivos e nos negativos. Uma feature que vale o mesmo nas duas classes não ajuda a separar nada, e o modelo corretamente a ignora.

É a confirmação, vinda de dentro do modelo, do que a seção 1 já dizia: achar `GT` é trivial, e toda a informação está no contexto ao redor.

## 3.7. Capacidade do modelo e tamanho de amostra

O modelo tem 805 parâmetros. Vale perguntar quanto dado ele precisa para sustentá-los.

In [ ]:
sorteio = np.random.default_rng(42)
indices_pos = np.flatnonzero(y_treino == 1)
indices_neg = np.flatnonzero(y_treino == 0)

print(f"{'positivos':>10} {'exemplos':>10} {'por parâmetro':>14} {'PR-AUC':>9}")
print("-" * 48)
for n_pos in (250, 500, 1000, 2500, 5000):
    escolhidos = np.concatenate([
        sorteio.choice(indices_pos, n_pos, replace=False),
        sorteio.choice(indices_neg, n_pos * 10, replace=False)])
    pequeno = LogisticRegression(max_iter=2000, n_jobs=-1)
    pequeno.fit(F_treino[escolhidos], y_treino[escolhidos])
    p = average_precision_score(y_teste, pequeno.decision_function(F_teste))
    print(f"{br(n_pos):>10} {br(len(escolhidos)):>10} "
          f"{br(len(escolhidos) / 805, 1):>14} {br(p, 4):>9}")

A curva sobe e depois achata. Com 250 positivos o modelo tem cerca de três exemplos por parâmetro, e não consegue estimar 805 pesos com isso. Com 5.000 positivos são quase setenta exemplos por parâmetro, e o ganho já é pequeno.

A leitura que vale levar adiante: capacidade não é boa nem ruim por si só. Ela rende quando há dado que a sustente, e desperdiça quando não há.

Guarde isso para a seção 5, onde o modelo mais caro da aula vai perder para este aqui.

# 4. Redes neurais convolucionais

A regressão logística trata cada posição da janela como independente. Ela não sabe que a posição 40 é vizinha da 41, e nada na estrutura dela diz isso.

Esta seção troca essa hipótese por outra, escolhida a partir de como o sinal biológico de fato se organiza.

## 4.1. Por que convolução em dados genômicos

Duas propriedades do dado, e cada uma vira uma escolha de arquitetura.

**O sinal é local e curto.** Sítio de splice, sítio de ligação de fator de transcrição, motivo regulatório: todos são padrões contíguos de poucas bases. Uma convolução com janela pequena embute essa vizinhança por construção, em vez de esperar que o modelo a descubra sozinho.

**O mesmo padrão aparece em posições diferentes.** Um filtro é um conjunto de pesos reaplicado em toda posição da sequência. O padrão é aprendido uma vez, e não uma vez por posição. Em genômica isso é a regra: motivos se repetem ao longo do genoma.

In [ ]:
from tensorflow import keras

# Quanto custa cada abordagem, em número de pesos.
conv = keras.Sequential([keras.layers.Input((201, 4)),
                         keras.layers.Conv1D(32, 9, activation="relu")])

print(f"regressão logística (804 features + intercepto): {br(805):>8} pesos")
print(f"camada convolucional, 32 filtros de largura 9:  {br(conv.count_params()):>8} pesos")
print()
print("A camada convolucional cobre as 201 posições com esses pesos,")
print("porque o mesmo filtro é reaplicado em cada uma delas.")

Um cuidado para não tirar a conclusão errada daí: compartilhar peso **não** significa que a rede inteira seja menor. A rede completa desta seção tem cerca de 50 mil parâmetros, contra 805 da logística. O compartilhamento vale
para a camada convolucional isolada, não para o modelo todo.

O argumento a favor da convolução não é economia. É que ela impõe uma estrutura compatível com o dado, e isso faz a capacidade extra render em vez de desperdiçar.

A evidência é medida. Variando a largura da janela e comparando os dois modelos:

| janela | logística | CNN |
|---|---|---|
| 21 nt | 0,388 | 0,422 |
| 41 nt | 0,483 | 0,530 |
| 81 nt | 0,592 | 0,655 |
| 201 nt | 0,647 | 0,726 |
| 401 nt | 0,636 | 0,673 |

A CNN ganha em toda janela testada, e a distância cresce com a largura. Quanto mais contexto se dá, mais a estrutura convolucional rende, enquanto o modelo linear satura e depois piora.

## 4.2. Agregação do mapa de ativação

Passar 32 filtros sobre uma janela de 201 bases produz um mapa de ativação: para cada filtro, o quanto ele casou em cada posição.

In [ ]:
mapa = keras.Sequential([keras.layers.Input((201, 4)),
                         keras.layers.Conv1D(32, 9, activation="relu")])
print("saída da convolução:", mapa.output_shape)
print()
print("193 posições onde o filtro coube, vezes 32 filtros.")
print("Falta transformar isso em um único número por janela.")

Há duas formas de resumir esse mapa, e elas fazem perguntas diferentes:

1. `GlobalMaxPooling`: para cada filtro, guardar o maior valor em qualquer posição. Pergunta: *este padrão aparece em algum lugar da janela?*
2. `MaxPooling` seguido de `Flatten`: reduzir a resolução mas manter a posição, e entregar tudo à camada densa. Pergunta: *este padrão aparece, e onde?*

Antes de treinar, decida qual das duas faz mais sentido aqui.

A pista está na seção 1: toda janela deste dataset, positiva ou negativa, tem `GT` na mesma posição 100.

Vale separar duas ideias que vêm juntas na mesma camada e são diferentes.

Compartilhar pesos entre posições é bom, e é o que torna a convolução eficiente: o filtro é o mesmo em qualquer lugar. Descartar a posição em que ele casou é outra coisa, e depende do problema.

## 4.3. Treino das duas variantes

As duas redes diferem em uma camada só. Todo o resto é igual.

A variante A usa `MaxPooling1D` seguido de `Flatten`, que reduz a resolução mas mantém a posição de cada casamento.

In [ ]:
# Semente fixada antes de construir: os pesos iniciais são sorteados aqui,
# e não no treino.
keras.utils.set_random_seed(42)

modelo_maxpooling = keras.Sequential([
    keras.layers.Input((201, 4)),
    keras.layers.Conv1D(32, 9, activation="relu"),
    keras.layers.MaxPooling1D(4),
    keras.layers.Flatten(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
], name="maxpooling")

modelo_maxpooling.summary()

A variante B troca `MaxPooling` mais `Flatten` por `GlobalMaxPooling`, e nada mais.

In [ ]:
keras.utils.set_random_seed(42)

modelo_globalmaxpooling = keras.Sequential([
    keras.layers.Input((201, 4)),
    keras.layers.Conv1D(32, 9, activation="relu"),
    keras.layers.GlobalMaxPooling1D(),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
], name="globalmaxpooling")

print(f"variante A, MaxPooling:       {br(modelo_maxpooling.count_params()):>7} parâmetros")
print(f"variante B, GlobalMaxPooling: {br(modelo_globalmaxpooling.count_params()):>7} parâmetros")

In [ ]:
import time

# Um pedaço da validação basta para a parada antecipada, e deixa a época rápida.
X_val_oh = one_hot_lote(dados["X_val"][:20000])
y_val = dados["y_val"][:20000]


def treinar(modelo):
    """Treina monitorando PR-AUC de validação, e não a perda.

    Com 50 negativos por positivo, a perda de validação é dominada pelos
    negativos e estabiliza cedo, enquanto o ordenamento dos positivos ainda
    está melhorando. Parar pela perda subtreina o modelo.
    """
    modelo.compile(optimizer="adam", loss="binary_crossentropy",
                   metrics=[keras.metrics.AUC(curve="PR", name="pr")])
    inicio = time.time()
    historico = modelo.fit(
        X_treino_oh, y_treino,
        validation_data=(X_val_oh, y_val),
        epochs=30, batch_size=256, verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_pr", mode="max", patience=5, restore_best_weights=True)])
    segundos = time.time() - inicio
    epocas = len(historico.history["loss"])
    tempos[modelo.name] = segundos
    print(f"{modelo.name:<18} {br(segundos, 0):>4} s, {epocas} épocas")
    return modelo.predict(X_teste_oh, verbose=0, batch_size=512).ravel()


score_maxpooling = treinar(modelo_maxpooling)
score_globalmaxpooling = treinar(modelo_globalmaxpooling)

## 4.4. Resultados

In [ ]:
print(f"{'modelo':<26} {'PR-AUC':>8} {'ROC-AUC':>9}")
print("-" * 46)
print(f"{'piso trivial':<26} {br(taxa_positivos, 4):>8} {'-':>9}")
print(f"{'regressão logística':<26} {br(pr_auc, 4):>8} {br(roc_auc, 4):>9}")
for nome, score in [("CNN com MaxPooling", score_maxpooling),
                    ("CNN com GlobalMaxPooling", score_globalmaxpooling)]:
    p = average_precision_score(y_teste, score)
    r = roc_auc_score(y_teste, score)
    print(f"{nome:<26} {br(p, 4):>8} {br(r, 4):>9}")

A variante A ganha da regressão logística. A variante B perde por um fator de dez e fica logo acima do piso trivial.

A diferença entre as duas é uma camada.

O mecanismo: o pooling global pergunta se o padrão aparece em algum lugar da janela. Mas positivos e negativos têm `GT` na mesma posição, e o que separa as classes é o contexto naquela posição específica. O pooling global descarta exatamente a informação que discrimina.

E quanto maior a janela, mais posições espúrias competem pelo máximo. Medido:

| janela | GlobalMaxPooling | MaxPooling |
|---|---|---|
| 21 nt | 0,366 | 0,422 |
| 41 nt | 0,243 | 0,530 |
| 81 nt | 0,146 | 0,655 |
| 201 nt | 0,057 | 0,726 |
| 401 nt | 0,039 | 0,673 |

Uma piora enquanto a outra melhora. Não é falta de treino: a variante B rodou o orçamento inteiro de épocas sem encostar na outra.

Vale notar que a variante B é **menor**, com 1.729 parâmetros contra 50.401. Ou seja, aqui o modelo mais enxuto é o pior. O que decide não é o tamanho, é a hipótese que a arquitetura assume sobre o dado.

Um aviso sobre os números que apareceram na sua tela. Rede neural tem inicialização aleatória, e mesmo com semente fixa o resultado varia entre máquinas e entre CPU e GPU. Medimos essa variação na preparação da aula: a variante A oscila cerca de 0,02 de PR-AUC entre execuções, tipicamente entre 0,66 e 0,72.

A regressão logística não tem essa variação, porque não depende de sorteio. Compare os números da turma: a diferença entre eles é pequena, e a diferença para a variante B não é.

## 4.5. ROC versus PR, revisitado

Na seção 3 ficou uma promessa: um caso em que as duas métricas discordam de forma gritante. A variante B é esse caso.

In [ ]:
pr_b = average_precision_score(y_teste, score_globalmaxpooling)
roc_b = roc_auc_score(y_teste, score_globalmaxpooling)

fpr_b, tpr_b, _ = roc_curve(y_teste, score_globalmaxpooling)
prec_b, rev_b, _ = precision_recall_curve(y_teste, score_globalmaxpooling)

figura, eixos = plt.subplots(1, 2, figsize=(11, 4.2), facecolor=SUPERFICIE)

eixos[0].plot([0, 1], [0, 1], linestyle="--", color=CINZA, linewidth=1.2)
eixos[0].plot(fpr_b, tpr_b, color=AZUL, linewidth=2)
eixos[0].set_title(f"ROC, AUC = {br(roc_b, 3)}", fontsize=11, color=TINTA, loc="left")
eixos[0].set_xlabel("taxa de falso positivo", fontsize=9, color=MUTED)
eixos[0].set_ylabel("taxa de verdadeiro positivo", fontsize=9, color=MUTED)
eixos[0].text(0.55, 0.42, "acaso", fontsize=9, color=CINZA, rotation=32)

eixos[1].axhline(taxa_positivos, linestyle="--", color=CINZA, linewidth=1.2)
eixos[1].plot(rev_b, prec_b, color=AZUL, linewidth=2)
eixos[1].set_title(f"Precisão-revocação, AUC = {br(pr_b, 3)}",
                   fontsize=11, color=TINTA, loc="left")
eixos[1].set_xlabel("revocação", fontsize=9, color=MUTED)
eixos[1].set_ylabel("precisão", fontsize=9, color=MUTED)
eixos[1].text(0.30, 0.62, f"linha tracejada = acaso ({br(taxa_positivos, 4)})",
              fontsize=9, color=CINZA)

for eixo in eixos:
    eixo.set_xlim(0, 1)
    eixo.set_ylim(0, 1)
    eixo.set_facecolor(SUPERFICIE)
    eixo.grid(True, color=GRADE, linewidth=0.8)
    eixo.set_axisbelow(True)
    for lado in ("top", "right"):
        eixo.spines[lado].set_visible(False)
    for lado in ("bottom", "left"):
        eixo.spines[lado].set_color(GRADE)
    eixo.tick_params(colors=MUTED, labelsize=8)

figura.suptitle("A variante que fracassou, vista por duas métricas",
                fontsize=12, color=TINTA, x=0.01, ha="left")
figura.tight_layout()
plt.show()

Um modelo quase inútil, que a curva ROC ainda faz parecer razoável.

Na seção 3 o contraste era brando, porque a logística é um modelo bom. Aqui não há como confundir: a PR-AUC está praticamente encostada no piso, e a ROC-AUC continua num valor que, num relatório, passaria sem ninguém perguntar nada.

## 4.6. Filtros aprendidos

Cada um dos 32 filtros da variante A é uma matriz `(9, 4)`: nove posições, quatro bases. É a mesma forma dos coeficientes da seção 3, só que bem menor.

Abaixo, os seis de maior magnitude.

In [ ]:
filtros = modelo_maxpooling.layers[0].get_weights()[0]   # (9, 4, 32)
normas = np.linalg.norm(filtros.reshape(-1, filtros.shape[-1]), axis=0)
escolhidos = np.argsort(-normas)[:6]

figura, eixos = plt.subplots(2, 3, figsize=(11, 4.0), facecolor=SUPERFICIE)
limite = np.abs(filtros[:, :, escolhidos]).max()

for eixo, k in zip(eixos.ravel(), escolhidos):
    filtro = filtros[:, :, k]
    eixo.imshow(filtro.T, cmap=DIVERGENTE, vmin=-limite, vmax=limite, aspect="auto")
    # Base de maior peso em cada uma das 9 posições do filtro.
    dominante = "".join(BASES[int(np.argmax(filtro[p]))] for p in range(9))
    eixo.set_title(f"filtro {k}:  {dominante}", fontsize=10, color=TINTA, loc="left")
    eixo.set_yticks(range(4))
    eixo.set_yticklabels(list(BASES), fontsize=8, color=TINTA)
    eixo.set_xticks(range(9))
    eixo.set_xticklabels(range(9), fontsize=7, color=MUTED)
    eixo.tick_params(length=0)
    for lado in eixo.spines.values():
        lado.set_visible(False)

figura.suptitle("Seis filtros aprendidos, e a base dominante em cada posição",
                fontsize=12, color=TINTA, x=0.01, ha="left")
figura.tight_layout()
plt.show()

Vários deles carregam pedaços do consenso. É comum aparecer `GTAAGT` inteiro, ou `AGGTAAGT` com o `AG` do lado do éxon incluído.

Nem todos são legíveis, e isso é honesto dizer: leitura de filtro é sugestiva, não é prova. Alguns vão parecer ruído, outros capturam combinações que não têm nome na literatura.

O que fecha a seção: dois modelos de naturezas bem diferentes, um linear e um convolucional, chegaram ao mesmo motivo biológico por caminhos distintos. Nenhum dos dois recebeu uma única linha de informação sobre biologia.

Na seção 5, uma arquitetura que lê a sequência inteira em ordem, para testar se ainda há sinal que estas duas não pegaram.

# 5. Redes neurais recorrentes

A convolução olha nove bases por vez e desliza. Cada filtro tem visão local, e a camada densa só recebe o resumo.

Esta seção testa a hipótese oposta: um modelo que lê a janela inteira, base por base, em ordem, guardando memória do que já viu.

## 5.1. Dependência de longo alcance

A promessa da rede recorrente é captar relação entre partes distantes da sequência. A posição 10 e a posição 190 nunca entram juntas num filtro de largura 9, mas entram na memória de uma recorrente que percorreu as duas.

E a promessa é razoável, porque splicing **é** um processo de longo alcance na biologia. O spliceossomo reconhece o par doador-aceitador, e esses dois podem estar separados por milhares de bases.

Vale deixar a hipótese escrita antes de testar:

> Se existir dependência de longo alcance dentro desta janela, a recorrente deve ganhar da convolução.

## 5.2. Arquitetura e treino

A arquitetura é pequena de propósito, para o custo vir da recorrência e não do tamanho.

Bidirecional porque não há razão para privilegiar um sentido de leitura: a janela tem éxon de um lado e íntron do outro, e os dois lados informam.

In [ ]:
keras.utils.set_random_seed(42)

modelo_lstm = keras.Sequential([
    keras.layers.Input((201, 4)),
    keras.layers.Bidirectional(keras.layers.LSTM(32)),
    keras.layers.Dense(1, activation="sigmoid"),
], name="lstm")

modelo_lstm.summary()

print(f"\nSão {br(modelo_lstm.count_params())} parâmetros, bem menos que os "
      f"{br(modelo_maxpooling.count_params())} da CNN da seção 4.")

Agora o treino.

Repare no orçamento: 60 épocas contra as 30 que a CNN teve, e paciência maior na parada antecipada. Não é descuido, é uma escolha.

A recorrente converge mais devagar, e com 30 épocas ela **ainda estava melhorando** quando o treino parava. Comparar um modelo interrompido no meio com outro que convergiu não diria nada sobre arquitetura, diria só sobre quem ganhou mais tempo. O orçamento maior é o que torna a comparação honesta.

A célula leva vários minutos, muito mais que tudo o que rodou até aqui. Dispare e leia a subseção 5.3 enquanto ela trabalha.

In [ ]:
inicio = time.time()

modelo_lstm.compile(optimizer="adam", loss="binary_crossentropy",
                    metrics=[keras.metrics.AUC(curve="PR", name="pr")])
historico_lstm = modelo_lstm.fit(
    X_treino_oh, y_treino,
    validation_data=(X_val_oh, y_val),
    epochs=60, batch_size=256, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_pr", mode="max", patience=8, restore_best_weights=True)])

tempos["lstm"] = time.time() - inicio
score_lstm = modelo_lstm.predict(X_teste_oh, verbose=0, batch_size=512).ravel()

print(f"\nTreinou em {br(tempos['lstm'], 0)} s, "
      f"{len(historico_lstm.history['loss'])} épocas")

## 5.3. Custo computacional da recorrência

A lentidão não é detalhe de implementação, é consequência da arquitetura.

Numa convolução, as 201 posições são processadas ao mesmo tempo. Cada uma só depende das nove bases embaixo do filtro, e nada impede que todas sejam calculadas de uma vez.

Numa recorrente, a posição 101 precisa do estado que saiu da posição 100, que precisou da 99, e assim por diante. São 201 passos que só podem acontecer em sequência.

Duas consequências práticas:

1. o custo por época cresce com o comprimento da janela, e não cai dividindo o trabalho entre mais núcleos;
2. GPU ajuda menos aqui do que numa convolução, porque o gargalo é a dependência entre passos, não a quantidade de contas.

É por isso que a mesma janela de 201 bases custa segundos numa CNN e minutos numa recorrente.

## 5.4. Resultados

In [ ]:
modelos = [
    ("regressão logística", modelo.decision_function(F_teste), "regressão logística"),
    ("CNN com MaxPooling", score_maxpooling, "maxpooling"),
    ("CNN com GlobalMaxPooling", score_globalmaxpooling, "globalmaxpooling"),
    ("LSTM bidirecional", score_lstm, "lstm"),
]

print(f"{'modelo':<26} {'PR-AUC':>8} {'ROC-AUC':>9} {'treino':>10}")
print("-" * 57)
print(f"{'piso trivial':<26} {br(taxa_positivos, 4):>8} {'-':>9} {'-':>10}")
for nome, score, chave in modelos:
    p = average_precision_score(y_teste, score)
    r = roc_auc_score(y_teste, score)
    print(f"{nome:<26} {br(p, 4):>8} {br(r, 4):>9} "
          f"{br(tempos[chave], 0):>7} s")

A LSTM perde para a CNN, e por uma margem clara.

Contra a regressão logística ela fica praticamente empatada. A diferença entre as duas é menor que a variação que a própria CNN apresenta entre execuções, ou seja, não dá para afirmar que uma é melhor que a outra.

O que não é ambíguo é o preço. Compare a última coluna: a recorrente custou dezenas de vezes mais tempo que os dois modelos com os quais empata ou perde.

E há um detalhe que precisa ser dito, porque muda a leitura: ela **não convergiu**. Bateu o teto de 60 épocas ainda melhorando.

In [ ]:
val_pr = historico_lstm.history["val_pr"]

figura, eixo = plt.subplots(figsize=(8, 3.2), facecolor=SUPERFICIE)
eixo.plot(range(1, len(val_pr) + 1), val_pr, color=AZUL, linewidth=2)
eixo.set_xlabel("época", fontsize=9, color=MUTED)
eixo.set_ylabel("PR-AUC de validação", fontsize=9, color=MUTED)
eixo.set_facecolor(SUPERFICIE)
eixo.grid(True, color=GRADE, linewidth=0.8)
eixo.set_axisbelow(True)
for lado in ("top", "right"):
    eixo.spines[lado].set_visible(False)
for lado in ("bottom", "left"):
    eixo.spines[lado].set_color(GRADE)
eixo.tick_params(colors=MUTED, labelsize=8)

figura.suptitle("A curva ainda estava subindo quando o orçamento acabou",
                fontsize=12, color=TINTA, x=0.01, ha="left")
figura.tight_layout()
plt.show()

A curva não achata. Com mais épocas ela provavelmente continuaria subindo, e é possível que em algum ponto alcançasse a CNN.

Essa é a resposta honesta: **não sabemos onde ela para**. E descobrir custaria mais tempo do que a resposta vale, o que é em si o resultado desta seção.

Vale registrar o contraste de orçamento, que não é pequeno:

| modelo | épocas até parar | parou por quê |
|---|---|---|
| CNN com MaxPooling | ~12 | convergiu, a validação parou de melhorar |
| LSTM bidirecional | 60 | acabou o orçamento, ainda melhorando |

## 5.5. Navalha de Occam

Vale voltar à hipótese que abriu a seção: se existir dependência de longo alcance **dentro desta janela**, a recorrente deve ganhar.

Ela não ganhou, e o motivo é que a hipótese não se aplica ao recorte. Dentro de 201 bases centradas no doador, o que existe é o motivo local mais um viés de composição distribuído, e a convolução pega os dois melhor e mais barato.

A dependência de longo alcance de verdade no splicing é entre doador e aceitador, separados por milhares de bases. Essa dependência **foi recortada fora** quando escolhemos a janela de 201 nt. Nenhum modelo captura o que o dado não contém.

In [ ]:
mediana_intron = 167   # medido no genoma do cupuaçu, na preparação da aula

print(f"Janela usada nesta aula:          {201:>7} bases")
print(f"Íntron mediano deste genoma:      {mediana_intron:>7} bases")
print(f"Íntron mais longo deste genoma:   {147316:>7} bases")
print()
print("O aceitador do íntron mediano já cai fora da janela.")
print("A dependência que a recorrente procuraria não está no dado.")

A navalha de Occam, em formato operacional: entre duas explicações que dão o mesmo resultado, prefira a mais simples.

Aqui o caso é mais concreto que o ditado. A recorrente levou dezenas de vezes mais tempo para empatar com um modelo de 805 parâmetros que roda em três segundos, e para perder de um que roda em dezessete.

O encerramento honesto, para ninguém sair com a lição errada. Nada disso significa que recorrente é ruim, nem que a regra é usar sempre CNN. Duas coisas mais específicas:

1. a arquitetura codifica uma hipótese sobre a estrutura do dado, e a hipótese da recorrência não se paga neste recorte;
2. custo faz parte da avaliação. Um modelo que talvez chegasse lá depois de uma hora não é comparável a um que chega em dezessete segundos.

Mude o recorte, com janela de dezenas de milhares de bases contendo doador e aceitador juntos, e a conclusão pode mudar junto. É exatamente o caminho que os modelos da seção 6 seguem.

# 6. PyTorch e modelos do estado da arte

Duas coisas para fechar.

A primeira é mostrar que a CNN da seção 4 não depende do Keras. A segunda é olhar um modelo de fronteira e reconhecer nele as peças construídas aqui.

## 6.1. A mesma arquitetura em PyTorch

Keras e PyTorch são as duas bibliotecas mais usadas para redes neurais. A escolha entre elas é de ecossistema e preferência, não de capacidade.

Abaixo, a mesma CNN da seção 4 escrita em PyTorch. Ela não vai ser treinada, só montada, para a comparação ficar concreta.

In [ ]:
import torch
from torch import nn

cnn_pytorch = nn.Sequential(
    nn.Conv1d(4, 32, 9), nn.ReLU(),
    nn.MaxPool1d(4),
    nn.Flatten(),
    nn.Linear(48 * 32, 32), nn.ReLU(),
    nn.Linear(32, 1), nn.Sigmoid(),
)

print(cnn_pytorch)

Camada por camada, a correspondência é direta:

| Keras | PyTorch |
|---|---|
| `keras.layers.Conv1D(32, 9)` | `nn.Conv1d(4, 32, 9)` |
| `keras.layers.MaxPooling1D(4)` | `nn.MaxPool1d(4)` |
| `keras.layers.Flatten()` | `nn.Flatten()` |
| `keras.layers.Dense(32)` | `nn.Linear(1536, 32)` |
| `keras.layers.Dense(1)` | `nn.Linear(32, 1)` |

Duas diferenças que aparecem já na primeira hora de quem troca de biblioteca:

1. PyTorch pede o tamanho de entrada de cada camada, Keras deduz sozinho. Por isso aparece `4` no `Conv1d` e `1536` no primeiro `Linear`.
2. A ordem dos eixos é trocada. Keras espera `(exemplos, comprimento, canais)`, PyTorch espera `(exemplos, canais, comprimento)`.

In [ ]:
entrada_keras = X_teste_oh[:2].shape
entrada_torch = tuple(torch.zeros(2, 4, 201).shape)

print(f"Keras espera:   {entrada_keras}   (exemplos, comprimento, canais)")
print(f"PyTorch espera: {entrada_torch}   (exemplos, canais, comprimento)")
print()
print("Trocar de biblioteca sem trocar a ordem dos eixos é o erro mais comum")
print("de quem faz a migração, e o mais chato de achar.")

E a prova de que é de fato a mesma arquitetura: as duas têm exatamente o mesmo número de pesos a ajustar.

In [ ]:
params_keras = modelo_maxpooling.count_params()
params_torch = sum(p.numel() for p in cnn_pytorch.parameters())

print(f"Keras:   {br(params_keras):>8} parâmetros")
print(f"PyTorch: {br(params_torch):>8} parâmetros")
print()
print("Iguais." if params_keras == params_torch else "Diferentes, algo não bate.")

O que falta no código acima é o laço de treino, que em PyTorch é escrito à mão: percorrer os lotes, calcular a perda, chamar o backward, atualizar os pesos. O Keras esconde tudo isso dentro do `.fit()`.

Nenhum dos dois é mais correto. Um é mais explícito, o outro é mais curto.

## 6.2. AlphaGenome

O AlphaGenome (Avsec et al., *Nature* 649:1206-1218, 2026) é um dos modelos de fronteira para predição de função a partir de sequência genômica.

<img src="images/alphagenome.png" width="720">

Percorrendo o diagrama, boa parte dos blocos já é conhecida:

| no diagrama | onde apareceu nesta aula |
|---|---|
| entrada de sequência one-hot A/C/G/T | seção 1 |
| blocos convolucionais | seção 4 |
| redução de resolução por pooling | seção 4 |
| cabeça de saída por tarefa | seções 3 a 5 |

O que é novo: os blocos de atenção, a escala da janela de entrada, e o fato de haver muitas cabeças de saída ao mesmo tempo em vez de uma só.

In [ ]:
linhas = [
    ("janela de entrada", "201 bases", "~1 milhão de bases"),
    ("saídas do modelo", "1 (é sítio?)", "milhares"),
    ("sítios cobertos", "doador", "doador e aceitador"),
    ("outros sinais", "nenhum", "expressão, e mais"),
    ("treino da CNN", br(tempos["maxpooling"], 0) + " s", "escala industrial"),
]

print(f"{'':<20} {'esta aula':>14}   {'AlphaGenome':<20}")
print("-" * 58)
for rotulo, aqui, la in linhas:
    print(f"{rotulo:<20} {aqui:>14}   {la:<20}")

A diferença é de escala e de recursos, não de conceito. Os blocos são os mesmos, e é por isso que dá para olhar a figura de um artigo da *Nature* e reconhecer o que está acontecendo.

## 6.3. Limites do recorte adotado

Terminar pelos limites é mais útil do que terminar por uma conclusão redonda. Três deles, e todos importam para quem for usar isso na própria pesquisa.

**O primeiro é um teto de revocação embutido no dataset.** Todo candidato aqui é um `GT`, então íntrons que começam com `GC` ou `AT` estão fora por construção. Nenhum modelo desta aula pode encontrá-los, por melhor que seja.

In [ ]:
canonicos = genoma["donor_sites"]
anotados = genoma["annotated_introns"]

print(f"Íntrons anotados no genoma:        {br(anotados):>9}")
print(f"Começam com GT, os únicos vistos:  {br(canonicos):>9}")
print(f"Teto de revocação desta aula:      {br(canonicos / anotados * 100, 2):>8}%")

**O segundo é a origem dos rótulos.** Eles vêm de anotação computacional apoiada em evidência transcriptômica, e não de validação experimental sítio a sítio. O teto real de desempenho é o teto do anotador, e nenhuma métrica desta aula enxerga esse limite.

**O terceiro é a formulação do problema.** SpliceAI e AlphaGenome não classificam janelas isoladas. Eles percorrem o cromossomo prevendo, para cada posição, se ali existe um sítio. É a formulação correta, e não cabia no
orçamento de uma aula.

Esse terceiro ponto fecha o laço com a seção 5. A dependência de longo alcance que a LSTM não encontrou é justamente a que só existe quando a janela é grande o bastante para conter doador e aceitador juntos. O problema não estava na arquitetura, estava no recorte.

### 6.3.1. Para seguir adiante

- Jaganathan et al., *Cell* 176:535-548, 2019. O SpliceAI, que é a versão
  completa do problema desta aula.
- Avsec et al., *Nature* 649:1206-1218, 2026. O AlphaGenome.
- [Alves et al., *GigaScience* 13:giae027, 2024](https://pubmed.ncbi.nlm.nih.gov/38837946/). O genoma de
  cupuaçu usado aqui, disponível no [GigaDB 102523](https://gigadb.org/dataset/102523).
- Athanasopoulou et al., *Curr. Issues Mol. Biol.* 47:470, 2025. Panorama de
  aprendizado de máquina em genômica.